In [1]:
!pip install -q faiss-cpu beir sentence-transformers transformers \
    rouge-score sacrebleu bert-score evaluate \
    matplotlib seaborn pandas scikit-learn tqdm scipy --upgrade

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 59.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.7/345.7 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 90.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.5/13.5 MB 81.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.7/37.7 M

In [2]:
import os, time, random, gc, warnings, math
from pathlib import Path
from typing import List, Dict, Tuple

import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns, faiss
from tqdm.auto import tqdm

import torch
from sklearn.decomposition import PCA
from sklearn.utils.extmath import randomized_svd
from sklearn.random_projection import GaussianRandomProjection
from sklearn.preprocessing import normalize
from sklearn.metrics import ndcg_score
from scipy.optimize import curve_fit

from beir import util
from beir.datasets.data_loader import GenericDataLoader
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSeq2SeqLM

from rouge_score import rouge_scorer
from bert_score import score as bert_score

from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings("ignore")
plt.style.use("ggplot")
sns.set_palette("Set2")
pd.options.display.float_format = "{:.4f}".format
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
RNG    = np.random.default_rng(42)


2025-05-13 11:08:49.085781: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1747134529.295944      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1747134529.354390      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [3]:
DATA_DIR = Path("datasets")
DATASET_URLS = {
    "scifact"     : "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/scifact.zip",
    "scidocs"  : "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/scidocs.zip",
    "fiqa" : "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/fiqa.zip",
    "nfcorpus"       : "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/nfcorpus.zip"
}


In [4]:
#Add Your Datasets to DATASET_LIST
DATASET_LIST = ["scifact","scidocs", "fiqa", "nfcorpus"]

In [5]:
def load_dataset(name, split="test"):
    url = DATASET_URLS[name]
    path = util.download_and_unzip(url, str(DATA_DIR))
    corpus, queries, qrels = GenericDataLoader(path).load(split=split)
    doc_ids = sorted(corpus.keys()); query_ids = sorted(queries.keys())
    docs   = [corpus[d]["text"] for d in doc_ids]
    qtexts = [queries[q] for q in query_ids]
    relevant=[]
    for qid in query_ids:
        idx=[doc_ids.index(d) for d in qrels.get(qid,{}) if d in doc_ids]
        relevant.append(np.asarray(idx,dtype=int))
    return {"docs":docs,"queries":qtexts,"relevant":relevant}

In [6]:
#Add Your Embedding Models to EMBED_MODELS
EMBED_MODELS = {
    "mpnet"  : "sentence-transformers/all-mpnet-base-v2"
}

embedders = {k: SentenceTransformer(v, device=DEVICE) for k,v in EMBED_MODELS.items()}
precomputed={}
for ds in DATASET_LIST:
    print(f"Embedding {ds}")
    data = load_dataset(ds)
    precomputed[ds] = {"docs":data["docs"],"queries":data["queries"],
                       "relevant":data["relevant"],"embs":{}}
    for eb, mdl in embedders.items():
        d_emb = mdl.encode(data["docs"], batch_size=128, show_progress_bar=False, convert_to_numpy=True)
        q_emb = mdl.encode(data["queries"], batch_size=128, show_progress_bar=False, convert_to_numpy=True)
        
        mu = d_emb.mean(axis = 0, keepdims = True)
        d_emb = d_emb - mu
        q_emb = q_emb - mu
        d_emb = normalize(d_emb); q_emb = normalize(q_emb)
        precomputed[ds]["embs"][eb] = {"doc_emb":d_emb,"qry_emb":q_emb}
        print(f"{eb}: {d_emb.shape}")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.4k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

  0%|          | 0/57638 [00:00<?, ?it/s]

  mpnet: (57638, 768)
Embedding nfcorpus


datasets/nfcorpus.zip:   0%|          | 0.00/2.34M [00:00<?, ?iB/s]

  0%|          | 0/3633 [00:00<?, ?it/s]

  mpnet: (3633, 768)


In [7]:
from sklearn.metrics.pairwise import cosine_similarity

def retrieve_topk_cosine(query_emb, doc_emb, k=10):
    scores = cosine_similarity(query_emb, doc_emb)
    return np.argsort(-scores, axis=1)[:, :k]

def faiss_lsh_index(doc_emb, n_bits=None):
    d = doc_emb.shape[1]
    if n_bits is None:
        n_bits = d * 2  # more bits = better accuracy
    index = faiss.IndexLSH(d, n_bits)
    faiss.normalize_L2(doc_emb)
    index.add(doc_emb)
    return index

def retrieve_lsh(index, query_emb, k=10):
    faiss.normalize_L2(query_emb)
    _, I = index.search(query_emb, k)
    return I

def faiss_pq_index(doc_emb, m=16, nbits=8):
    d = doc_emb.shape[1]
    quantizer = faiss.IndexFlatL2(d)
    index = faiss.IndexPQ(d, m, nbits)
    index.train(doc_emb)
    index.add(doc_emb)
    return index

def retrieve_pq(index, query_emb, k=10):
    _, I = index.search(query_emb, k)
    return I

def recall_at_k(pred_indices, ground_truth, k=10):
    correct = 0
    for i in range(len(ground_truth)):
        gt_set = set(ground_truth[i])
        pred_set = set(pred_indices[i][:k])
        correct += len(gt_set & pred_set)
    return correct / (len(ground_truth) * k)

def cosine_distortion(orig_emb, comp_emb, n_samples=1000, seed=42):
    rng = np.random.default_rng(seed)
    n = orig_emb.shape[0]
    idx_i = rng.integers(0, n, size=n_samples)
    idx_j = rng.integers(0, n, size=n_samples)
    orig = normalize(orig_emb)
    comp = normalize(comp_emb)
    cos_orig = np.sum(orig[idx_i] * orig[idx_j], axis=1)
    cos_comp = np.sum(comp[idx_i] * comp[idx_j], axis=1)
    return np.mean(np.abs(cos_orig - cos_comp))

def knn_overlap_at_k(orig_emb, comp_emb, k=10):
    from sklearn.metrics.pairwise import cosine_similarity
    sim_orig = cosine_similarity(orig_emb)
    sim_comp = cosine_similarity(comp_emb)
    overlap = []
    for i in range(sim_orig.shape[0]):
        orig_top = np.argsort(-sim_orig[i])[:k]
        comp_top = np.argsort(-sim_comp[i])[:k]
        overlap.append(len(set(orig_top) & set(comp_top)) / k)
    return np.mean(overlap)

def precision_at_k(pred_indices, gt_indices, k=10):
    precisions = []
    for preds, gts in zip(pred_indices, gt_indices):
        precisions.append(len(set(preds[:k]) & set(gts)) / k)
    return np.mean(precisions)

def mrr_at_k(pred_indices, gt_indices, k=10):
    rr = []
    for preds, gts in zip(pred_indices, gt_indices):
        rank = next((i+1 for i,doc in enumerate(preds[:k]) if doc in gts), None)
        rr.append(1.0/rank if rank else 0.0)
    return np.mean(rr)

In [8]:
def randsvd_Q(M, k, p=10, q=2, device=None):
    if device is None:
        device = M.device

    m, n = M.shape
    l = k + p

    Omega = torch.randn(n, l, device=device) 
    Y = M @ Omega 
    for _ in range(q): 
        Y = M @ (M.T @ Y)

    Q, _ = torch.linalg.qr(Y, mode='reduced')

    return Q

def randsvd_decompose_and_project(docs_np, queries_np, k, p=10, q=2, device='cuda'):
    M = torch.from_numpy(docs_np).to(device)  
    
    Qr = randsvd_Q(M.T, k, p=p, q=q, device=device)
    
    Qr_k = Qr[:, :k]  # (d, k)
    
    docs_rsvd = (M @ Qr_k).cpu().numpy()          
    queries_rsvd = queries_np @ Qr_k.cpu().numpy()  
    
    return docs_rsvd, queries_rsvd

In [13]:
import time
import numpy as np
import pandas as pd
from sklearn.utils.extmath import randomized_svd
from sklearn.random_projection import GaussianRandomProjection
import faiss

def evaluate_all_methods_with_energy(docs, queries, gt_top, k=10, k_svd=256, p=10, q=2, rp_dim=256, device="cuda"):
    methods = ["Vanilla","FullSVD","RandSVD-PI","RandSVD","GaussianRP","LSH","PQ"]
    results = {m: {} for m in methods}

    orig_fro_sq = np.sum(docs**2)

    #Vanilla
    t0 = time.time()
    preds_v = retrieve_topk_cosine(queries, docs, k)
    t1 = time.time()
    results["Vanilla"] = {
        "Recall":          recall_at_k(preds_v, gt_top, k),
        "Precision":       precision_at_k(preds_v, gt_top, k),
        "MRR":             mrr_at_k(preds_v, gt_top, k),
        "CosineDist":      0.0,
        "kNN@k":           1.0,
        "EnergyRetained":  np.sum(docs**2)/orig_fro_sq,  # everything retained
        "preprocess":      0.0,
        "search":          t1 - t0
    }

    #Full SVD
    t0 = time.time()
    U,S,VT = np.linalg.svd(docs, full_matrices=False)
    VT_k = VT[:k_svd]
    docs_svd    = docs @ VT_k.T
    queries_svd = queries @ VT_k.T
    prep = time.time()-t0
    t0 = time.time()
    preds = retrieve_topk_cosine(queries_svd, docs_svd, k)
    search = time.time()-t0
    results["FullSVD"] = {
        "Recall":       recall_at_k(preds, gt_top, k),
        "Precision":    precision_at_k(preds, gt_top, k),
        "MRR":          mrr_at_k(preds, gt_top, k),
        "CosineDist":   cosine_distortion(docs, docs_svd),
        "kNN@k":        knn_overlap_at_k(docs, docs_svd, k),
        "EnergyRetained": np.sum(docs_svd**2)/orig_fro_sq,
        "preprocess":   prep,
        "search":       search
    }

    #RandSVD‐PI
    t0 = time.time()
    docs_rsvd_qr, queries_rsvd_qr = randsvd_decompose_and_project(
        docs, queries, k=k_svd, p=p, q=q, device=device
    )
    prep = time.time()-t0
    t0 = time.time()
    preds = retrieve_topk_cosine(queries_rsvd_qr, docs_rsvd_qr, k)
    search = time.time()-t0
    results["RandSVD-PI"] = {
        "Recall":       recall_at_k(preds, gt_top, k),
        "Precision":    precision_at_k(preds, gt_top, k),
        "MRR":          mrr_at_k(preds, gt_top, k),
        "CosineDist":   cosine_distortion(docs, docs_rsvd_qr),
        "kNN@k":        knn_overlap_at_k(docs, docs_rsvd_qr, k),
        "EnergyRetained": np.sum(docs_rsvd_qr**2)/orig_fro_sq,
        "preprocess":   prep,
        "search":       search
    }

    #RandSVD
    t0 = time.time()
    docs_rsvd_sk, queries_rsvd_sk = randsvd_decompose_and_project(
        docs, queries, k=k_svd, p=p, q=0, device=device
    )
    prep = time.time()-t0
    t0 = time.time()
    preds = retrieve_topk_cosine(queries_rsvd_sk, docs_rsvd_sk, k)
    search = time.time()-t0
    results["RandSVD"] = {
        "Recall":       recall_at_k(preds, gt_top, k),
        "Precision":    precision_at_k(preds, gt_top, k),
        "MRR":          mrr_at_k(preds, gt_top, k),
        "CosineDist":   cosine_distortion(docs, docs_rsvd_sk),
        "kNN@k":        knn_overlap_at_k(docs, docs_rsvd_sk, k),
        "EnergyRetained": np.sum(docs_rsvd_sk**2)/orig_fro_sq,
        "preprocess":   prep,
        "search":       search
    }

    #Gaussian Random Projection
    t0 = time.time()
    rp = GaussianRandomProjection(n_components=rp_dim, random_state=0)
    docs_rp = rp.fit_transform(docs)
    queries_rp = rp.transform(queries)
    prep = time.time()-t0
    t0 = time.time()
    preds = retrieve_topk_cosine(queries_rp, docs_rp, k)
    search = time.time()-t0
    results["GaussianRP"] = {
        "Recall":       recall_at_k(preds, gt_top, k),
        "Precision":    precision_at_k(preds, gt_top, k),
        "MRR":          mrr_at_k(preds, gt_top, k),
        "CosineDist":   cosine_distortion(docs, docs_rp),
        "kNN@k":        knn_overlap_at_k(docs, docs_rp, k),
        "EnergyRetained": np.sum(docs_rp**2)/orig_fro_sq,
        "preprocess":   prep,
        "search":       search
    }

    #LSH
    t0 = time.time()
    lsh_idx = faiss.IndexLSH(docs.shape[1], docs.shape[1]*4)
    lsh_idx.add(docs.astype('float32'))
    prep = time.time()-t0
    t0 = time.time()
    preds = retrieve_lsh(lsh_idx, queries, k)
    search = time.time()-t0
    results["LSH"] = {
        "Recall":       recall_at_k(preds, gt_top, k),
        "Precision":    precision_at_k(preds, gt_top, k),
        "MRR":          mrr_at_k(preds, gt_top, k),
        "CosineDist":   0.0,
        "kNN@k":        1.0,
        "EnergyRetained": np.sum(docs**2)/orig_fro_sq,
        "preprocess":   prep,
        "search":       search
    }

    #Product Quantization
    t0 = time.time()
    pq_idx = faiss.IndexPQ(docs.shape[1], 64, 8)
    pq_idx.train(docs.astype('float32'))
    pq_idx.add(docs.astype('float32'))
    docs_pq = np.vstack([pq_idx.reconstruct(i) for i in range(docs.shape[0])])
    prep = time.time()-t0
    t0 = time.time()
    preds = retrieve_pq(pq_idx, queries, k)
    search = time.time()-t0
    results["PQ"] = {
        "Recall":       recall_at_k(preds, gt_top, k),
        "Precision":    precision_at_k(preds, gt_top, k),
        "MRR":          mrr_at_k(preds, gt_top, k),
        "CosineDist":   cosine_distortion(docs, docs_pq),
        "kNN@k":        knn_overlap_at_k(docs, docs_pq, k),
        "EnergyRetained": np.sum(docs_pq**2)/orig_fro_sq,
        "preprocess":   prep,
        "search":       search
    }

    df = pd.DataFrame(results).T
    df = df[[
      "Recall","CosineDist",
      "kNN@k","EnergyRetained","preprocess","search"
    ]]
    return df


**SciDocs Queries on SciFact Corpora**

In [14]:
d = "scifact" #Set To Whichever Corpora You Need
q = "scidocs" #Set To Whichever Queries You Need
docs   = precomputed[d]["embs"]["mpnet"]["doc_emb"]
queries= precomputed[q]["embs"]["mpnet"]["qry_emb"]
gt_top = retrieve_topk_cosine(queries, docs, k=10)
df_all = evaluate_all_methods_with_energy(docs, queries, gt_top, k=10)
df_all


WARNING clustering 5183 points to 256 centroids: please provide at least 9984 training points
WARNING clustering 5183 points to 256 centroids: please provide at least 9984 training points
WARNING clustering 5183 points to 256 centroids: please provide at least 9984 training points
WARNING clustering 5183 points to 256 centroids: please provide at least 9984 training points
WARNING clustering 5183 points to 256 centroids: please provide at least 9984 training points
WARNING clustering 5183 points to 256 centroids: please provide at least 9984 training points
WARNING clustering 5183 points to 256 centroids: please provide at least 9984 training points
WARNING clustering 5183 points to 256 centroids: please provide at least 9984 training points
WARNING clustering 5183 points to 256 centroids: please provide at least 9984 training points
WARNING clustering 5183 points to 256 centroids: please provide at least 9984 training points
WARNING clustering 5183 points to 256 centroids: please prov

,Recall,CosineDist,kNN@k,EnergyRetained,preprocess,search
Vanilla,1.0000,0.0000,1.0000,1.0000,0.0000,0.1802
FullSVD,0.8212,0.0064,0.9706,0.9458,0.5458,0.1539
RandSVD-PI,0.8113,0.0069,0.9673,0.9423,0.0141,0.1505
RandSVD,0.6795,0.0172,0.9244,0.8526,0.0118,0.1479
GaussianRP,0.2991,0.0498,0.7800,0.9901,0.0207,0.1490
LSH,0.6643,0.0000,1.0000,1.0000,1.0170,0.1169
PQ,0.5246,0.0311,0.7744,0.6734,1.1217,0.1461


**SciFact Queries on SciFact Corpora**

In [15]:
d = "scifact" #Set To Whichever Corpora You Need
q = "scifact" #Set To Whichever Queries You Need
docs   = precomputed[d]["embs"]["mpnet"]["doc_emb"]
queries= precomputed[q]["embs"]["mpnet"]["qry_emb"]
gt_top = retrieve_topk_cosine(queries, docs, k=10)
df_all = evaluate_all_methods_with_energy(docs, queries, gt_top, k=10)
df_all


WARNING clustering 5183 points to 256 centroids: please provide at least 9984 training points
WARNING clustering 5183 points to 256 centroids: please provide at least 9984 training points
WARNING clustering 5183 points to 256 centroids: please provide at least 9984 training points
WARNING clustering 5183 points to 256 centroids: please provide at least 9984 training points
WARNING clustering 5183 points to 256 centroids: please provide at least 9984 training points
WARNING clustering 5183 points to 256 centroids: please provide at least 9984 training points
WARNING clustering 5183 points to 256 centroids: please provide at least 9984 training points
WARNING clustering 5183 points to 256 centroids: please provide at least 9984 training points
WARNING clustering 5183 points to 256 centroids: please provide at least 9984 training points
WARNING clustering 5183 points to 256 centroids: please provide at least 9984 training points
WARNING clustering 5183 points to 256 centroids: please prov

,Recall,CosineDist,kNN@k,EnergyRetained,preprocess,search
Vanilla,1.0000,0.0000,1.0000,1.0000,0.0000,0.0622
FullSVD,0.9487,0.0064,0.9706,0.9458,0.5324,0.0501
RandSVD-PI,0.9440,0.0070,0.9672,0.9424,0.0130,0.0482
RandSVD,0.8860,0.0169,0.9247,0.8527,0.0102,0.0454
GaussianRP,0.6680,0.0498,0.7800,0.9901,0.0196,0.0477
LSH,0.8433,0.0000,1.0000,1.0000,0.9951,0.0395
PQ,0.7713,0.0311,0.7744,0.6734,1.1476,0.0320


**SciDocs Queries on SciDocs Corpora**

In [16]:
d = "scidocs" #Set To Whichever Corpora You Need
q = "scidocs" #Set To Whichever Queries You Need
docs   = precomputed[d]["embs"]["mpnet"]["doc_emb"]
queries= precomputed[q]["embs"]["mpnet"]["qry_emb"]
gt_top = retrieve_topk_cosine(queries, docs, k=10)
df_all = evaluate_all_methods_with_energy(docs, queries, gt_top, k=10)
df_all


,Recall,CosineDist,kNN@k,EnergyRetained,preprocess,search
Vanilla,1.0000,0.0000,1.0000,1.0000,0.0000,1.0474
FullSVD,0.9239,0.0095,0.9277,0.9049,2.4699,0.8813
RandSVD-PI,0.9178,0.0105,0.9228,0.8985,0.0438,0.8789
RandSVD,0.8618,0.0221,0.8758,0.7777,0.0336,0.9053
GaussianRP,0.7043,0.0501,0.7538,1.0003,0.0686,0.8824
LSH,0.8515,0.0000,1.0000,1.0000,1.4737,0.7352
PQ,0.7466,0.0301,0.6973,0.6314,5.5350,0.4656
